# Deep Learning Image Search dengan CLIP

## 1. Setup Path Project

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Import Modul Deep Embedding

In [ ]:
from src.cbir.deep_embedding import (
    DEFAULT_CLIP_MODEL,
    build_and_save_deep_embedding_index,
    evaluate_deep_embedding_index_file,
    search_deep_embedding_index_file,
)
from src.cbir.visualization import show_search_results

DEFAULT_CLIP_MODEL

## 3. Konfigurasi Dataset dan Index

In [ ]:
IMAGE_DIR = PROJECT_ROOT / "data" / "images"
INDEX_PATH = PROJECT_ROOT / "models" / "index-clip.pkl"
TOP_K = 10
BATCH_SIZE = 16
DEVICE = "auto"

IMAGE_DIR, INDEX_PATH

## 4. Build Deep Embedding Index

In [ ]:
index = build_and_save_deep_embedding_index(
    image_dir=IMAGE_DIR,
    index_path=INDEX_PATH,
    model_name=DEFAULT_CLIP_MODEL,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

print("Indexed images:", len(index.image_paths))
print("Embedding shape:", index.embeddings.shape)
print("Build seconds:", round(index.build_seconds, 2))

## 5. Evaluasi Precision@10

In [ ]:
summary = evaluate_deep_embedding_index_file(
    index_path=INDEX_PATH,
    top_k=TOP_K,
    device=DEVICE,
)

print("Evaluated queries:", summary.query_count)
print(f"Mean precision@{summary.top_k}:", round(summary.mean_precision, 4))

## 6. Test Query Satu Gambar

In [ ]:
QUERY_IMAGE = IMAGE_DIR / "Borobudur-Temple" / "Borobudur.jpg"

response = search_deep_embedding_index_file(
    query_image_path=QUERY_IMAGE,
    index_path=INDEX_PATH,
    top_k=TOP_K,
    device=DEVICE,
)

print("Query seconds:", round(response.query_seconds, 4))
for position, result in enumerate(response.results, start=1):
    print(f"{position:02d}. distance={result.distance:.6f} | {result.image_path}")

## 7. Visualisasi Hasil Query

In [ ]:
show_search_results(
    query_image_path=QUERY_IMAGE,
    results=response.results,
)